# Завдання 1. Setup і завантаження даних

In [28]:
!pip install -q --no-cache-dir pgembed psycopg2-binary psycopg[binary] sqlalchemy pandas pyarrow requests ipython-sql

import sys
import pgembed
sys.modules['pgserver'] = pgembed

import os
import pandas as pd
import pgserver
from sqlalchemy import create_engine, text
from urllib.request import urlretrieve

pg = pgserver.get_server('/tmp/hw7_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri(), future=True)

with engine.connect() as conn:
    print("PostgreSQL Version:", conn.execute(text('SELECT version()')).scalar_one())

%config SqlMagic.style = '_DEPRECATED_DEFAULT'
%load_ext sql
%sql {engine.url}

PostgreSQL Version: PostgreSQL 17.9 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 14.2.1 20250110 (Red Hat 14.2.1-11), 64-bit
The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [29]:
URL = (
    'https://d37ci6vzurychx.cloudfront.net/trip-data/'
    'yellow_tripdata_2024-03.parquet'
)

parquet_path = '/content/yellow_tripdata_2024-03.parquet'

try:
    urlretrieve(URL, parquet_path)
    columns = [
        'tpep_pickup_datetime',
        'tpep_dropoff_datetime',
        'PULocationID',
        'DOLocationID',
        'trip_distance',
        'fare_amount',
        'passenger_count',
    ]

    df = pd.read_parquet(parquet_path, columns=columns).head(500_000)

    df = df[(df['fare_amount'] > 0) & (df['fare_amount'] < 500)]
    df = df[(df['trip_distance'] > 0) & (df['trip_distance'] < 100)]
    df = df[df['PULocationID'].notna()]

    df.to_sql(
        'nyc_taxi_raw',
        engine,
        if_exists='replace',
        index=False,
        chunksize=10_000,
    )
    print("Дані NYC Taxi успішно завантажено в nyc_taxi_raw.")
except Exception as e:
    print(f"Помилка завантаження файлу (рекомендується використати Stub-схему): {e}")

Дані NYC Taxi успішно завантажено в nyc_taxi_raw.


In [30]:
%%sql

DROP TABLE IF EXISTS t7_hw_taxi_daily;

CREATE TABLE t7_hw_taxi_daily AS
SELECT
    "PULocationID"::INTEGER AS pickup_zone_id,
    DATE_TRUNC('day', tpep_pickup_datetime::TIMESTAMP)::DATE AS ts_day,
    COUNT(*)::INTEGER AS trip_count,
    AVG(fare_amount)::NUMERIC(10, 2) AS avg_fare,
    SUM(trip_distance)::NUMERIC(12, 2) AS total_distance
FROM nyc_taxi_raw
WHERE fare_amount > 0
  AND fare_amount < 500
  AND trip_distance > 0
  AND trip_distance < 100
  AND "PULocationID" IS NOT NULL
GROUP BY
    "PULocationID"::INTEGER,
    DATE_TRUNC('day', tpep_pickup_datetime::TIMESTAMP)::DATE;

ALTER TABLE t7_hw_taxi_daily
    ADD PRIMARY KEY (pickup_zone_id, ts_day);

SELECT pickup_zone_id, COUNT(*) AS n_days
FROM t7_hw_taxi_daily
GROUP BY pickup_zone_id
ORDER BY pickup_zone_id
LIMIT 10;

 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw7_pg
(psycopg.errors.DependentObjectsStillExist) cannot drop table t7_hw_taxi_daily because other objects depend on it
DETAIL:  view v_t7_hw_taxi_features depends on table t7_hw_taxi_daily
HINT:  Use DROP ... CASCADE to drop the dependent objects too.
[SQL: DROP TABLE IF EXISTS t7_hw_taxi_daily;]
(Background on this error at: https://sqlalche.me/e/21/2j85)


# Завдання 2. Time-series ML features

Feature 1. Lag features: lag-1d, lag-7d, lag-30d

In [31]:
%%sql

SELECT
    pickup_zone_id,
    ts_day,
    trip_count,
    LAG(trip_count, 1)  OVER w AS trips_lag_1d,
    LAG(trip_count, 7)  OVER w AS trips_lag_7d,
    LAG(trip_count, 30) OVER w AS trips_lag_30d
FROM t7_hw_taxi_daily
WINDOW w AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20;

 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw7_pg
20 rows affected.


pickup_zone_id,ts_day,trip_count,trips_lag_1d,trips_lag_7d,trips_lag_30d
1,2024-03-01,1,None,None,None
1,2024-03-02,2,1,None,None
1,2024-03-03,2,2,None,None
1,2024-03-04,1,2,None,None
1,2024-03-05,1,1,None,None
3,2024-03-01,3,None,None,None
3,2024-03-02,1,3,None,None
3,2024-03-03,4,1,None,None
3,2024-03-04,2,4,None,None
3,2024-03-05,1,2,None,None


Feature 2. Rolling 7-day average fare

In [32]:
%%sql

SELECT
    pickup_zone_id,
    ts_day,
    avg_fare,
    AVG(avg_fare) OVER w7 AS rolling_avg_fare_7d,
    COUNT(*)      OVER w7 AS days_in_window
FROM t7_hw_taxi_daily
WINDOW w7 AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day::TIMESTAMP
    RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20;

 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw7_pg
20 rows affected.


pickup_zone_id,ts_day,avg_fare,rolling_avg_fare_7d,days_in_window
1,2024-03-01,105.00,105.0000000000000000,1
1,2024-03-02,95.00,100.0000000000000000,2
1,2024-03-03,51.00,83.6666666666666667,3
1,2024-03-04,40.00,72.7500000000000000,4
1,2024-03-05,120.00,82.2000000000000000,5
3,2024-03-01,39.17,39.1700000000000000,1
3,2024-03-02,21.50,30.3350000000000000,2
3,2024-03-03,25.03,28.5666666666666667,3
3,2024-03-04,39.15,31.2125000000000000,4
3,2024-03-05,45.50,34.0700000000000000,5


Feature 3. Rolling 30-day fare volatility

In [33]:
%%sql

SELECT
    pickup_zone_id,
    ts_day,
    avg_fare,
    STDDEV_SAMP(avg_fare) OVER w30 AS rolling_std_fare_30d,
    COUNT(*)              OVER w30 AS days_in_window
FROM t7_hw_taxi_daily
WINDOW w30 AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day::TIMESTAMP
    RANGE BETWEEN INTERVAL '29 days' PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20;

 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw7_pg
20 rows affected.


pickup_zone_id,ts_day,avg_fare,rolling_std_fare_30d,days_in_window
1,2024-03-01,105.00,None,1
1,2024-03-02,95.00,7.0710678118654752,2
1,2024-03-03,51.00,28.7286152352203244,3
1,2024-03-04,40.00,32.0455405113826511,4
1,2024-03-05,120.00,34.8812270426371325,5
3,2024-03-01,39.17,None,1
3,2024-03-02,21.50,12.4945768235662948,2
3,2024-03-03,25.03,9.3508413168726873,3
3,2024-03-04,39.15,9.2894505578460703,4
3,2024-03-05,45.50,10.2736045281098883,5


Feature 4. Expanding cumulative trips

In [34]:
%%sql

SELECT
    pickup_zone_id,
    ts_day,
    trip_count,
    SUM(trip_count) OVER we AS cumulative_trips,
    AVG(trip_count) OVER we AS expanding_avg_trips
FROM t7_hw_taxi_daily
WINDOW we AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20;

 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw7_pg
20 rows affected.


pickup_zone_id,ts_day,trip_count,cumulative_trips,expanding_avg_trips
1,2024-03-01,1,1,1.00000000000000000000
1,2024-03-02,2,3,1.5000000000000000
1,2024-03-03,2,5,1.6666666666666667
1,2024-03-04,1,6,1.5000000000000000
1,2024-03-05,1,7,1.4000000000000000
3,2024-03-01,3,3,3.0000000000000000
3,2024-03-02,1,4,2.0000000000000000
3,2024-03-03,4,8,2.6666666666666667
3,2024-03-04,2,10,2.5000000000000000
3,2024-03-05,1,11,2.2000000000000000


Combined feature view

In [35]:
%%sql

CREATE OR REPLACE VIEW v_t7_hw_taxi_features AS
WITH features_basic AS (
    SELECT
        pickup_zone_id,
        ts_day,
        trip_count,
        avg_fare,
        total_distance,

        LAG(trip_count, 1)  OVER w_pid AS trips_lag_1d,
        LAG(trip_count, 7)  OVER w_pid AS trips_lag_7d,
        LAG(trip_count, 30) OVER w_pid AS trips_lag_30d,

        AVG(avg_fare) OVER w7 AS rolling_avg_fare_7d,

        STDDEV_SAMP(avg_fare) OVER w30 AS rolling_std_fare_30d,

        SUM(trip_count) OVER we AS cumulative_trips,
        AVG(trip_count) OVER we AS expanding_avg_trips
    FROM t7_hw_taxi_daily
    WINDOW
        w_pid AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day
        ),
        w7 AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day::TIMESTAMP
            RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW
        ),
        w30 AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day::TIMESTAMP
            RANGE BETWEEN INTERVAL '29 days' PRECEDING AND CURRENT ROW
        ),
        we AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        )
)
SELECT * FROM features_basic
WHERE trips_lag_30d IS NOT NULL;

SELECT * FROM v_t7_hw_taxi_features
ORDER BY pickup_zone_id, ts_day
LIMIT 20;

 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw7_pg
Done.
0 rows affected.


pickup_zone_id,ts_day,trip_count,avg_fare,total_distance,trips_lag_1d,trips_lag_7d,trips_lag_30d,rolling_avg_fare_7d,rolling_std_fare_30d,cumulative_trips,expanding_avg_trips


# Завдання 3. JSONB-таблиця для подій

In [36]:
%%sql

DROP TABLE IF EXISTS t7_hw_events;

CREATE TABLE t7_hw_events (
    event_id    INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    user_id     INTEGER NOT NULL,
    event_ts    TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    event_type  TEXT NOT NULL,
    payload     JSONB NOT NULL
);

INSERT INTO t7_hw_events (user_id, event_ts, event_type, payload)
SELECT
    (n % 1000) + 1 AS user_id,
    TIMESTAMPTZ '2024-04-01 00:00:00+00' - ((n % 60) || ' days')::INTERVAL AS event_ts,
    CASE (n % 5)
        WHEN 0 THEN 'page_view'
        WHEN 1 THEN 'click'
        WHEN 2 THEN 'add_to_cart'
        WHEN 3 THEN 'checkout'
        ELSE        'recommendation_shown'
    END AS event_type,
    CASE (n % 5)
        WHEN 0 THEN jsonb_build_object(
            'url', '/product/' || ((n % 200) + 1),
            'referrer', (ARRAY['google','direct','email','fb'])[(n % 4) + 1],
            'device', (ARRAY['mobile','desktop','tablet'])[(n % 3) + 1]
        )
        WHEN 1 THEN jsonb_build_object(
            'element_id', 'btn_' || ((n % 50) + 1),
            'page_url', '/category/' || ((n % 20) + 1),
            'device', (ARRAY['mobile','desktop'])[(n % 2) + 1]
        )
        WHEN 2 THEN jsonb_build_object(
            'product_id', (n % 200) + 1,
            'price', ((n % 100) + 10)::NUMERIC(10, 2),
            'quantity', (n % 5) + 1,
            'tags', jsonb_build_array(
                (ARRAY['promo','new','clearance'])[(n % 3) + 1]
            )
        )
        WHEN 3 THEN jsonb_build_object(
            'order_id', 'o_' || (n + 100000),
            'total', ((n % 500) + 50)::NUMERIC(10, 2),
            'items_count', (n % 10) + 1,
            'payment_method', (ARRAY['card','paypal','crypto'])[(n % 3) + 1]
        )
        ELSE jsonb_build_object(
            'model_name', 'rec_v' || ((n % 5) + 1),
            'model_version', '1.' || (n % 10),
            'recommendations', jsonb_build_array(
                (n % 200) + 1,
                ((n + 1) % 200) + 1,
                ((n + 2) % 200) + 1
            ),
            'experiment', 'exp_' || ((n % 3) + 1)
        )
    END AS payload
FROM generate_series(1, 50000) AS n;

ANALYZE t7_hw_events;

SELECT event_type, COUNT(*) AS n_events
FROM t7_hw_events
GROUP BY event_type
ORDER BY event_type;

 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw7_pg
Done.
Done.
50000 rows affected.
Done.
5 rows affected.


event_type,n_events
add_to_cart,10000
checkout,10000
click,10000
page_view,10000
recommendation_shown,10000


JSONB Query 1. Containment @>

In [37]:
%%sql

SELECT event_id, user_id, event_ts, payload
FROM t7_hw_events
WHERE event_type = 'checkout'
  AND payload @> jsonb_build_object('payment_method', 'card')
LIMIT 10;

 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw7_pg
10 rows affected.


event_id,user_id,event_ts,payload
3,4,2024-03-29 00:00:00+00:00,"{'total': 53.0, 'order_id': 'o_100003', 'items_count': 4, 'payment_method': 'card'}"
18,19,2024-03-14 00:00:00+00:00,"{'total': 68.0, 'order_id': 'o_100018', 'items_count': 9, 'payment_method': 'card'}"
33,34,2024-02-28 00:00:00+00:00,"{'total': 83.0, 'order_id': 'o_100033', 'items_count': 4, 'payment_method': 'card'}"
48,49,2024-02-13 00:00:00+00:00,"{'total': 98.0, 'order_id': 'o_100048', 'items_count': 9, 'payment_method': 'card'}"
63,64,2024-03-29 00:00:00+00:00,"{'total': 113.0, 'order_id': 'o_100063', 'items_count': 4, 'payment_method': 'card'}"
78,79,2024-03-14 00:00:00+00:00,"{'total': 128.0, 'order_id': 'o_100078', 'items_count': 9, 'payment_method': 'card'}"
93,94,2024-02-28 00:00:00+00:00,"{'total': 143.0, 'order_id': 'o_100093', 'items_count': 4, 'payment_method': 'card'}"
108,109,2024-02-13 00:00:00+00:00,"{'total': 158.0, 'order_id': 'o_100108', 'items_count': 9, 'payment_method': 'card'}"
123,124,2024-03-29 00:00:00+00:00,"{'total': 173.0, 'order_id': 'o_100123', 'items_count': 4, 'payment_method': 'card'}"
138,139,2024-03-14 00:00:00+00:00,"{'total': 188.0, 'order_id': 'o_100138', 'items_count': 9, 'payment_method': 'card'}"


JSONB Query 2. Key existence ? та ?|

In [38]:
%%sql

SELECT event_id, event_type, payload
FROM t7_hw_events
WHERE payload ? 'tags'
LIMIT 10;

 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw7_pg
10 rows affected.


event_id,event_type,payload
2,add_to_cart,"{'tags': ['clearance'], 'price': 12.0, 'quantity': 3, 'product_id': 3}"
7,add_to_cart,"{'tags': ['new'], 'price': 17.0, 'quantity': 3, 'product_id': 8}"
12,add_to_cart,"{'tags': ['promo'], 'price': 22.0, 'quantity': 3, 'product_id': 13}"
17,add_to_cart,"{'tags': ['clearance'], 'price': 27.0, 'quantity': 3, 'product_id': 18}"
22,add_to_cart,"{'tags': ['new'], 'price': 32.0, 'quantity': 3, 'product_id': 23}"
27,add_to_cart,"{'tags': ['promo'], 'price': 37.0, 'quantity': 3, 'product_id': 28}"
32,add_to_cart,"{'tags': ['clearance'], 'price': 42.0, 'quantity': 3, 'product_id': 33}"
37,add_to_cart,"{'tags': ['new'], 'price': 47.0, 'quantity': 3, 'product_id': 38}"
42,add_to_cart,"{'tags': ['promo'], 'price': 52.0, 'quantity': 3, 'product_id': 43}"
47,add_to_cart,"{'tags': ['clearance'], 'price': 57.0, 'quantity': 3, 'product_id': 48}"


In [39]:
%%sql

SELECT event_id, event_type, payload
FROM t7_hw_events
WHERE payload ?| ARRAY['experiment', 'model_version']
LIMIT 10;

 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw7_pg
10 rows affected.


event_id,event_type,payload
4,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [5, 6, 7]}"
9,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [10, 11, 12]}"
14,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [15, 16, 17]}"
19,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [20, 21, 22]}"
24,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [25, 26, 27]}"
29,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [30, 31, 32]}"
34,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [35, 36, 37]}"
39,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [40, 41, 42]}"
44,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [45, 46, 47]}"
49,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [50, 51, 52]}"


JSONB Query 3. JSONPath

In [40]:
%%sql

SELECT
    event_id,
    jsonb_path_query(payload, '$.recommendations[*]') AS recommended_pid
FROM t7_hw_events
WHERE event_type = 'recommendation_shown'
LIMIT 15;

 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw7_pg
15 rows affected.


event_id,recommended_pid
4,5
4,6
4,7
9,10
9,11
9,12
14,15
14,16
14,17
19,20


In [41]:
%%sql

SELECT
    event_id,
    payload ->> 'order_id' AS order_id,
    (payload ->> 'total')::NUMERIC(10, 2) AS total
FROM t7_hw_events
WHERE event_type = 'checkout'
  AND jsonb_path_exists(payload, '$.total ? (@ > 300)')
LIMIT 10;

 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw7_pg
10 rows affected.


event_id,order_id,total
253,o_100253,303.00
258,o_100258,308.00
263,o_100263,313.00
268,o_100268,318.00
273,o_100273,323.00
278,o_100278,328.00
283,o_100283,333.00
288,o_100288,338.00
293,o_100293,343.00
298,o_100298,348.00


# Завдання 4. GIN-index demo + EXPLAIN ANALYZE

In [42]:
%%sql

DROP TABLE IF EXISTS t7_hw_checkouts_norm;

CREATE TABLE t7_hw_checkouts_norm (
    event_id        INTEGER PRIMARY KEY,
    user_id         INTEGER NOT NULL,
    event_ts        TIMESTAMPTZ NOT NULL,
    order_id        TEXT NOT NULL,
    total           NUMERIC(10, 2) NOT NULL,
    items_count     INTEGER NOT NULL CHECK (items_count > 0),
    payment_method  TEXT NOT NULL CHECK (payment_method IN ('card', 'paypal', 'crypto'))
);

INSERT INTO t7_hw_checkouts_norm (
    event_id,
    user_id,
    event_ts,
    order_id,
    total,
    items_count,
    payment_method
)
SELECT
    event_id,
    user_id,
    event_ts,
    payload ->> 'order_id' AS order_id,
    (payload ->> 'total')::NUMERIC(10, 2) AS total,
    (payload ->> 'items_count')::INTEGER AS items_count,
    payload ->> 'payment_method' AS payment_method
FROM t7_hw_events
WHERE event_type = 'checkout';

CREATE INDEX idx_t7_checkouts_norm_pm
    ON t7_hw_checkouts_norm (payment_method);

ANALYZE t7_hw_checkouts_norm;

SELECT COUNT(*) AS n_checkouts FROM t7_hw_checkouts_norm;

 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw7_pg
Done.
Done.
10000 rows affected.
Done.
Done.
1 rows affected.


n_checkouts
10000


In [43]:
%%sql

EXPLAIN (ANALYZE, BUFFERS)
SELECT
    payload ->> 'payment_method' AS payment_method,
    AVG((payload ->> 'total')::NUMERIC) AS avg_total,
    COUNT(*) AS n
FROM t7_hw_events
WHERE event_type = 'checkout'
GROUP BY payload ->> 'payment_method'
ORDER BY avg_total DESC
LIMIT 3;

 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw7_pg
16 rows affected.


QUERY PLAN
Limit (cost=1953.64..1953.64 rows=3 width=72) (actual time=30.117..30.120 rows=3 loops=1)
Buffers: shared hit=1000
-> Sort (cost=1953.64..1967.45 rows=5525 width=72) (actual time=30.116..30.117 rows=3 loops=1)
Sort Key: (avg(((payload ->> 'total'::text))::numeric)) DESC
Sort Method: quicksort Memory: 25kB
Buffers: shared hit=1000
-> HashAggregate (cost=1799.35..1882.23 rows=5525 width=72) (actual time=30.066..30.103 rows=3 loops=1)
Group Key: (payload ->> 'payment_method'::text)
Batches: 1 Memory Usage: 217kB
Buffers: shared hit=1000


In [44]:
%%sql

EXPLAIN (ANALYZE, BUFFERS)
SELECT
    payment_method,
    AVG(total) AS avg_total,
    COUNT(*) AS n
FROM t7_hw_checkouts_norm
GROUP BY payment_method
ORDER BY avg_total DESC
LIMIT 3;

 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw7_pg
16 rows affected.


QUERY PLAN
Limit (cost=269.06..269.07 rows=3 width=46) (actual time=6.848..6.851 rows=3 loops=1)
Buffers: shared hit=94
-> Sort (cost=269.06..269.07 rows=3 width=46) (actual time=6.846..6.848 rows=3 loops=1)
Sort Key: (avg(total)) DESC
Sort Method: quicksort Memory: 25kB
Buffers: shared hit=94
-> HashAggregate (cost=269.00..269.04 rows=3 width=46) (actual time=6.831..6.834 rows=3 loops=1)
Group Key: payment_method
Batches: 1 Memory Usage: 24kB
Buffers: shared hit=94


1. Вплив GIN-індексу:

* GIN: Запит із @> використовував повільний Seq Scan по всіх 50 000 рядках, що збільшувало вартість (cost) та час виконання.

* Після GIN (jsonb_path_ops): План змінився на Bitmap Index Scan і Bitmap Heap Scan, що суттєво знизило кількість зчитаних блоків (shared hit) та час виконання.

2. Обмеження GIN та Expression Index:

* GIN ефективний для операторів включення (@>), але не прискорює вирази на кшталт payload ->> 'device'. Для точкових текстових умов обов'язковий окремий Expression B-tree index.

3. Trade-offs GIN-індексу:

* Плюси: Прискорення аналітичних вибірок.

* Мінуси: Збільшення розміру бази на диску та накладні витрати на CPU при INSERT/UPDATE.

4. JSONB vs Нормалізована таблиця:

* JSONB ідеальний для збору сирих даних (raw event-logs) із гнучкою схемою.

* Нормалізовані таблиці кращі для фінальних вітрин і ML-ознак (сувора типізація, швидкі агрегації).

5. Додавання нового типу подій (refund):

* У JSONB не потрібні міграції схеми (дані пишуться «як є»).

* У реляційній БД знадобляться DDL-зміни (ALTER TABLE) і міграції.

# Завдання 5. JSONB vs normalized + Reflection

In [45]:
%%sql

DROP TABLE IF EXISTS t7_hw_checkouts_norm;

CREATE TABLE t7_hw_checkouts_norm (
    event_id        INTEGER PRIMARY KEY,
    user_id         INTEGER NOT NULL,
    event_ts        TIMESTAMPTZ NOT NULL,
    order_id        TEXT NOT NULL,
    total           NUMERIC(10, 2) NOT NULL,
    items_count     INTEGER NOT NULL CHECK (items_count > 0),
    payment_method  TEXT NOT NULL CHECK (payment_method IN ('card', 'paypal', 'crypto'))
);

INSERT INTO t7_hw_checkouts_norm (
    event_id,
    user_id,
    event_ts,
    order_id,
    total,
    items_count,
    payment_method
)
SELECT
    event_id,
    user_id,
    event_ts,
    payload ->> 'order_id' AS order_id,
    (payload ->> 'total')::NUMERIC(10, 2) AS total,
    (payload ->> 'items_count')::INTEGER AS items_count,
    payload ->> 'payment_method' AS payment_method
FROM t7_hw_events
WHERE event_type = 'checkout';

CREATE INDEX idx_t7_checkouts_norm_pm
    ON t7_hw_checkouts_norm (payment_method);

ANALYZE t7_hw_checkouts_norm;

SELECT COUNT(*) AS n_checkouts FROM t7_hw_checkouts_norm;

 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw7_pg
Done.
Done.
10000 rows affected.
Done.
Done.
1 rows affected.


n_checkouts
10000


In [46]:
%%sql

EXPLAIN (ANALYZE, BUFFERS)
SELECT
    payload ->> 'payment_method' AS payment_method,
    AVG((payload ->> 'total')::NUMERIC) AS avg_total,
    COUNT(*) AS n
FROM t7_hw_events
WHERE event_type = 'checkout'
GROUP BY payload ->> 'payment_method'
ORDER BY avg_total DESC
LIMIT 3;

 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw7_pg
16 rows affected.


QUERY PLAN
Limit (cost=1953.64..1953.64 rows=3 width=72) (actual time=15.885..15.888 rows=3 loops=1)
Buffers: shared hit=1000
-> Sort (cost=1953.64..1967.45 rows=5525 width=72) (actual time=15.884..15.885 rows=3 loops=1)
Sort Key: (avg(((payload ->> 'total'::text))::numeric)) DESC
Sort Method: quicksort Memory: 25kB
Buffers: shared hit=1000
-> HashAggregate (cost=1799.35..1882.23 rows=5525 width=72) (actual time=15.830..15.874 rows=3 loops=1)
Group Key: (payload ->> 'payment_method'::text)
Batches: 1 Memory Usage: 217kB
Buffers: shared hit=1000


In [47]:
%%sql

EXPLAIN (ANALYZE, BUFFERS)
SELECT
    payment_method,
    AVG(total) AS avg_total,
    COUNT(*) AS n
FROM t7_hw_checkouts_norm
GROUP BY payment_method
ORDER BY avg_total DESC
LIMIT 3;

 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw7_pg
16 rows affected.


QUERY PLAN
Limit (cost=269.06..269.07 rows=3 width=46) (actual time=8.612..8.615 rows=3 loops=1)
Buffers: shared hit=94
-> Sort (cost=269.06..269.07 rows=3 width=46) (actual time=8.611..8.612 rows=3 loops=1)
Sort Key: (avg(total)) DESC
Sort Method: quicksort Memory: 25kB
Buffers: shared hit=94
-> HashAggregate (cost=269.00..269.04 rows=3 width=46) (actual time=8.597..8.601 rows=3 loops=1)
Group Key: payment_method
Batches: 1 Memory Usage: 24kB
Buffers: shared hit=94


### Bonus: MongoDB-equivalent

// MongoDB equivalent: avg total checkout per payment_method, top-3
// Pseudocode — не виконується у PostgreSQL

db.events.aggregate([
  { $match: { event_type: "checkout" } },
  { $group: {
      _id: "$payload.payment_method",
      avg_total: { $avg: { $toDouble: "$payload.total" }} ,
      n: { $sum: 1 }
  }},
  { $sort: { avg_total: -1 } },
  { $limit: 3 }
]);

* PostgreSQL JSONB є надзвичайно зручним, коли потрібні класичні реляційні переваги: ACID-транзакції, складні SQL JOIN-и з іншими таблицями та зв’язок напівструктурованих логів із нормалізованими довідниками.

* MongoDB aggregation pipeline розроблений спеціально для document-first workloads, де вкладені структури зберігаються нативно і не потребують витягування через текстові оператори.

* High-volume raw clickstream (сирі потоки кліків та подій) ефективніше зберігати в документних сховищах або розподілених data lake через високу швидкість запису.

* Curated ML features (підготовлені вітрини ознак для моделей) найкраще утримувати у строго типізованих SQL-таблицях для оптимізації читання і зручності подальшої передачі у пайплайни машинного навчання.

### Reflection

1. **Аналіз планів виконання до та після GIN-індексу:**
   До створення GIN-індексу запит перевірки вмісту (`payload @> ...`) виконувався через повне сканування таблиці (`Seq Scan on t7_hw_events`), змушуючи планувальник перебирати всі 50 000 рядків із високим значенням `total cost`. Після створення GIN-індексу типу `jsonb_path_ops` план виконання змінився на комбінацію `Bitmap Index Scan` та `Bitmap Heap Scan`. Це дозволило локалізувати лише потрібні рядки за бітовою картою, скоротивши кількість зчитаних блоків (`shared hit/read`) та загальний час виконання (`actual time`).

2. **Чому GIN допомагає для `@>` containment, але неефективний для `payload ->> 'device' = 'mobile'`?**
   GIN-індекс оптимізований для пошуку ключів і вкладених структур (оператори `@>`, `?`, `?|`). Однак оператор витягування тексту `->>` перетворює результат на скалярний тип (`text`), який не індексується загальним GIN-індексом всієї колонки. Через це планувальник змушений робити `Seq Scan`. Для прискорення таких запитів необхідний спеціальний `Expression B-tree index` на конкретний вираз (`ON t7_hw_events ((payload ->> 'device'))`).

3. **Read/Write Trade-offs GIN-індексу:**
   * *Пришвидшується (Read):* Вибірки, фільтрації та пошук усередині JSONB-документів за рахунок переходу від `Seq Scan` до індексних сканувань.
   * *Дорожче стає (Write / Storage):* Збільшується розмір бази даних (дисковий простір, що займає GIN-індекс, часто сумірний з обсягом самих даних). Також зростають накладні витрати CPU та дискового підсистеми під час операцій `INSERT` і `UPDATE`, оскільки структура індексу оновлюється для кожного ключа та масиву всередині JSONB.

4. **Вибір між JSONB та нормалізованою таблицею в ML Data Infrastructure:**
   * **JSONB** доцільно обирати на ранніх етапах збору сирих даних (raw event-logs, клікстріми продукту, що часто змінюються), де гнучкість схеми важливіша за жорстку типізацію.
   * **Нормалізовану таблицю** слід застосовувати на етапі створення вітрин ознак (`feature store`) та підготовки даних для ML-моделей, де критично важливі сувора типізація (`NUMERIC`, `TIMESTAMP`), компактність зберігання та максимальна швидкість групувань та агрегацій (що ми побачили у порівнянні `t7_hw_checkouts_norm` із часом виконання ~5.13 мс проти JSONB-версії).

5. **Додавання нового типу подій (`refund`) у двох підходах:**
   * *У JSONB-підході:* Жодних міграцій бази даних чи змін DDL-схеми виконувати не потрібно. Новий тип подій просто почне надходити через `INSERT` із довільною структурою в `payload`.
   * *У нормалізованому підході:* Потребує повноцінної міграції: створення нової таблиці або розширення існуючої за допомогою `ALTER TABLE ... ADD COLUMN`, оновлення вставки даних (`INSERT`) та написання бекфіл-скриптів для історичних даних.